# M0 · 주제와 API 확정

**주제:** 산업재해 공개 데이터로 업종·규모 대시보드를 만들고, 사고사례·공식 문서를 근거로 답하는 RAG를 만든다.

**이 단계의 완료 기준:** ①~⑦ 출처와 원본을 추적할 수 있고, 필수 ②~⑤ API가 키를 사용한 소량 호출에서 실제로 응답하는지 확인한다. ⑥은 RAG 문서의 위치 목록으로 함께 검사한다. 이 노트북은 데이터를 정제하거나 위험률을 계산하지 않는다.

실행 전: VS Code에서 `C:\SANUP-P`를 열고 `.venv` Python 커널을 선택한다. 실제 키는 `.env`에만 둔다. 아래 코드는 키 값을 출력·저장하지 않는다.

## 1. 프로젝트와 수집 자료를 연결하기

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import zipfile
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import pandas as pd
import requests
from dotenv import dotenv_values
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None, 'VS Code에서 SANUP-P 프로젝트 폴더를 열고 이 노트북을 실행하세요.'
RAW = ROOT / 'data' / 'personal' / 'raw'
REPORTS = ROOT / 'data' / 'personal' / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
SECRETS = dotenv_values(ROOT / '.env') if (ROOT / '.env').exists() else {}
print('프로젝트:', ROOT)
print('원본 폴더:', RAW)

In [ ]:
SOURCES = [
    {'id': '01', 'dataset_id': '15140383', 'role': 'SIF 사례 분석·RAG 후보',
     'file': 'source_01_sif_archive.xlsx', 'encoding': None, 'key_env': None, 'uuid': None},
    {'id': '02', 'dataset_id': '15084672', 'role': '사고재해자 대시보드',
     'file': 'source_02_accident_injured.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_02_ACCIDENT_INJURED_API_KEY', 'uuid': 'uddi:8a4523d3-816b-4eb0-ad2f-9d88d5895e71'},
    {'id': '03', 'dataset_id': '15084674', 'role': '사고사망자 대시보드',
     'file': 'source_03_accident_deaths.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_03_ACCIDENT_DEATHS_API_KEY', 'uuid': 'uddi:6b3e32f7-6431-4a2f-978a-e207b058ae1d'},
    {'id': '04', 'dataset_id': '15064487', 'role': '사업장 수 대시보드',
     'file': 'source_04_establishments.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_04_ESTABLISHMENTS_API_KEY', 'uuid': 'uddi:d8b9a405-5ec5-445f-b1d1-99484da54212'},
    {'id': '05', 'dataset_id': '15064491', 'role': '사망만인율 대시보드',
     'file': 'source_05_fatality_rate.csv', 'encoding': 'cp949',
     'key_env': 'SOURCE_05_FATALITY_RATE_API_KEY', 'uuid': 'uddi:041d50b2-e50a-4bdd-95d9-bae6bdf4cee2'},
    {'id': '06', 'dataset_id': '15162988', 'role': 'RAG 원문 위치 목록',
     'file': 'source_06_rag_source_index.csv', 'encoding': 'utf-8-sig',
     'key_env': 'SOURCE_06_RAG_SOURCE_INDEX_API_KEY', 'uuid': 'uddi:6aad2c14-2c8e-4381-bc3b-9397ffe99a03'},
    {'id': '07', 'dataset_id': '15161288', 'role': 'M7 용어사전',
     'file': 'source_07_safety_terms.zip', 'encoding': None, 'key_env': None, 'uuid': None},
]
for source in SOURCES:
    source['page_url'] = f"https://www.data.go.kr/data/{source['dataset_id']}/fileData.do"
    source['api_url'] = (f"https://api.odcloud.kr/api/{source['dataset_id']}/v1/{source['uuid']}"
                         if source['uuid'] else None)

display(pd.DataFrame([{
    '번호': s['id'], '용도': s['role'], '원본 파일': s['file'],
    'API 방식': 'OpenAPI' if s['api_url'] else '파일 다운로드',
    'API 키 변수': s['key_env'] or '없음',
} for s in SOURCES]))

**출처:** [① SIF](https://www.data.go.kr/data/15140383/fileData.do) · [② 사고재해자](https://www.data.go.kr/data/15084672/fileData.do) · [③ 사고사망자](https://www.data.go.kr/data/15084674/fileData.do) · [④ 사업장 수](https://www.data.go.kr/data/15064487/fileData.do) · [⑤ 사망만인율](https://www.data.go.kr/data/15064491/fileData.do) · [⑥ RAG 원문 위치](https://www.data.go.kr/data/15162988/fileData.do) · [⑦ 용어사전](https://www.data.go.kr/data/15161288/fileData.do).

`fileData.do`는 자료 소개·다운로드 페이지다. 아래 API 주소는 별도로 확인된 `api.odcloud.kr`의 해당 자료 경로다. ①·⑦은 이번 프로젝트에서 파일로 확보했으므로 API 키가 필요 없다.

## 2. 원본 무결성·형식 확인

In [ ]:
manifest_path = RAW / 'download_manifest.csv'
assert manifest_path.exists(), '원본 다운로드 기록이 없습니다.'
with manifest_path.open(encoding='utf-8-sig', newline='') as stream:
    manifest = {row['source_id']: row for row in csv.DictReader(stream)}

file_checks = []
for s in SOURCES:
    p = RAW / f"source_{s['id']}" / s['file']
    item = manifest[s['id']]
    assert p.is_file(), f"원본 누락: {p.name}"
    digest = hashlib.sha256(p.read_bytes()).hexdigest().upper()
    assert digest == item['sha256'].upper(), f"원본 해시 불일치: {p.name}"
    check = {'source_id': s['id'], 'file': p.name, 'bytes': p.stat().st_size,
             'sha256_ok': True, 'format': p.suffix.lower(), 'rows': None, 'columns': None}
    if s['encoding']:
        frame = pd.read_csv(p, encoding=s['encoding'])
        check['rows'], check['columns'] = len(frame), list(frame.columns)
    elif s['id'] == '01':
        check['sheets'] = pd.ExcelFile(p).sheet_names
    elif s['id'] == '07':
        with zipfile.ZipFile(p) as archive:
            check['zip_members'] = [n for n in archive.namelist() if n.lower().endswith('.csv')]
    file_checks.append(check)

display(pd.DataFrame([{k: c[k] for k in ('source_id', 'file', 'bytes', 'sha256_ok', 'format', 'rows')}
                      for c in file_checks]))
print('① 시트:', file_checks[0]['sheets'])
print('⑦ ZIP 내부 CSV:', file_checks[-1]['zip_members'])

## 3. API를 2행씩 실제 호출해 인증·응답·열을 확인

In [ ]:
api_checks = []
session = requests.Session()
for s in SOURCES:
    if not s['api_url']:
        continue
    result = {'source_id': s['id'], 'dataset_id': s['dataset_id'], 'api_url': s['api_url'],
              'key_env': s['key_env'], 'http_status': None, 'total_count': None,
              'sample_rows': 0, 'schema_matches_file': False, 'status': 'not_checked',
              'error_code': None, 'error_message': None}
    key = (SECRETS.get(s['key_env']) or '').strip()
    if not key:
        result['status'] = 'key_missing'
    else:
        try:
            response = session.get(s['api_url'], params={'page': 1, 'perPage': 2, 'serviceKey': key}, timeout=20)
            result['http_status'] = response.status_code
            if response.ok:
                payload = response.json()
                rows = payload.get('data') if isinstance(payload, dict) else None
                if isinstance(rows, list) and all(isinstance(row, dict) for row in rows):
                    result['sample_rows'] = len(rows)
                    result['total_count'] = payload.get('totalCount')
                    local_columns = set(next(c['columns'] for c in file_checks if c['source_id'] == s['id']))
                    result['schema_matches_file'] = bool(rows) and set(rows[0]) == local_columns
                    result['status'] = 'ok' if result['schema_matches_file'] else 'schema_mismatch'
                else:
                    result['status'] = 'invalid_payload'
            else:
                result['status'] = 'http_error'
                try:
                    error_payload = response.json()
                    result['error_code'] = error_payload.get('code')
                    message = str(error_payload.get('msg') or error_payload.get('message') or '')[:240]
                    result['error_message'] = message.replace(key, '[REDACTED]')
                except ValueError:
                    result['error_message'] = 'Non-JSON HTTP error response'
        except (requests.RequestException, ValueError) as exc:
            # 예외 문자열이나 요청 URL에는 키가 들어갈 수 있어 유형만 기록한다.
            result['status'] = type(exc).__name__
    api_checks.append(result)

display(pd.DataFrame([{k: r[k] for k in ('source_id', 'http_status', 'total_count',
                                          'sample_rows', 'schema_matches_file', 'status',
                                          'error_code', 'error_message')}
                      for r in api_checks]))

## 4. 키를 제외한 검증 기록 저장

In [ ]:
report = {
    'checked_at': datetime.now(ZoneInfo('Asia/Seoul')).isoformat(timespec='seconds'),
    'subject': '산업재해 분석 대시보드와 근거 기반 RAG',
    'file_checks': file_checks,
    'api_checks': api_checks,
    'required_api_ids': ['02', '03', '04', '05', '06'],
    'latest_sif_comparison': json.loads((REPORTS / 'sif_latest_compare.json').read_text(encoding='utf-8')) if (REPORTS / 'sif_latest_compare.json').is_file() else None,
}
output = REPORTS / 'm0_source_validation.json'
output.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
required = [r for r in api_checks if r['source_id'] in report['required_api_ids']]
print('M0 필수 API 통과:', len(required) == len(report['required_api_ids']) and all(r['status'] == 'ok' for r in required))
print('검증 기록:', output)

## ⑥ 실제 보고서와 ⑧ 사용자 ZIP 확인

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None
raw = ROOT / 'data' / 'personal' / 'raw'
guides = raw / 'source_08'
zip_paths = [guides / 'kosha_1.zip', guides / 'kosha_2.zip']
assert all(p.is_file() for p in zip_paths)
assert all((guides / name).is_file() for name in ('README.md', 'kosha_guide_index.csv'))
guide_index = pd.read_csv(guides / 'kosha_guide_index.csv')
expected = {f'{row.guide_id}.pdf': str(row.sha256).lower() for row in guide_index.itertuples(index=False)}
archive_pdf_checks = []
seen = set()
zip_checks = []
for archive_path in zip_paths:
    archive_digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    with zipfile.ZipFile(archive_path) as archive:
        members = [name for name in archive.namelist() if name.lower().endswith('.pdf')]
        zip_checks.append({'file': archive_path.name, 'bytes': archive_path.stat().st_size,
                           'sha256': archive_digest, 'pdf_count': len(members)})
        for member in members:
            name = Path(member).name
            assert name in expected, f'색인에 없는 PDF: {name}'
            assert name not in seen, f'ZIP 중복 PDF: {name}'
            seen.add(name)
            archived_bytes = archive.read(member)
            extracted_path = guides / 'pdf' / name
            extracted_bytes = extracted_path.read_bytes()
            digest = hashlib.sha256(archived_bytes).hexdigest().lower()
            extracted_digest = hashlib.sha256(extracted_bytes).hexdigest().lower()
            archive_pdf_checks.append({'file': name, 'archive_sha256_matches_index': digest == expected[name],
                                       'extracted_matches_archive': digest == extracted_digest})
assert len(expected) == 18 and seen == set(expected)
assert all(row['archive_sha256_matches_index'] and row['extracted_matches_archive'] for row in archive_pdf_checks)
reports = raw / 'source_06' / 'reports'
report_index = pd.read_csv(reports / 'manifest.csv')
report_checks = []
for row in report_index.itertuples(index=False):
    pdf = reports / row.filename
    digest = hashlib.sha256(pdf.read_bytes()).hexdigest() if pdf.is_file() else None
    report_checks.append({'doc_id': row.doc_id, 'sha256_ok': digest is not None and digest.upper() == str(row.sha256).upper()})
assert len(report_checks) == 3 and all(row['sha256_ok'] for row in report_checks)
extra = {'source_06_reports': report_checks, 'source_08_zip_archives': zip_checks,
         'source_08_archive_pdfs': archive_pdf_checks, 'source_08_guides': [
             {'guide_id': row.guide_id, 'sha256_ok': hashlib.sha256((guides / 'pdf' / f'{row.guide_id}.pdf').read_bytes()).hexdigest().lower() == str(row.sha256).lower()}
             for row in guide_index.itertuples(index=False)]}
target = ROOT / 'data' / 'personal' / 'reports' / 'm0_extra_sources_validation.json'
target.write_text(json.dumps(extra, ensure_ascii=False, indent=2), encoding='utf-8')
print('⑥ 보고서 PDF:', len(report_checks), '건 해시 확인')
print('⑧ ZIP:', len(zip_checks), '개 · PDF:', len(archive_pdf_checks), '개, ZIP 안 원본·압축 해제본·색인 해시 일치')


## M0 마무리 기록 — 2026-10-03

### 이번 단계에서 정한 것

산업재해 공개자료로 업종·규모 대시보드를 만들고, 사고사례와 공식 안전지침을 근거로 답하는 RAG를 만든다. 자료 역할은 이렇게 나눴다.

- ① SIF 아카이브: 사고사례 분포·위험요인 분석과 RAG 후보
- ②~⑤: 업종×사업장 규모 대시보드 통계
- ⑥: 사고조사보고서·중대재해 사이렌의 **위치 목록**. 목록 CSV 자체를 사고 원문으로 취급하지 않는다.
- ⑦: 현장 용어와 법률 용어를 잇는 검색어 확장 실험
- ⑧: SIF와 함께 보여줄 공식 KOSHA GUIDE 안전지침

### 이번에 실제 확인한 것

- ①~⑦의 로컬 원본 7개가 `download_manifest.csv`의 SHA-256과 일치했다. ① XLSX에는 안내 시트와 제조업 등·건설업 시트가 있고, ②~⑤ CSV는 각각 30행, ⑥은 5행이다. ⑦ ZIP에는 CSV 2개가 있다.
- ②~⑥은 `.env`에서 키를 읽어 시험 호출했고 2026-10-03 재검증에서 모두 HTTP 200으로 응답했다. ③도 다른 API와 같은 공통 키로 정상 호출됐다. 키 값은 노트북·보고서에 기록하지 않았다.
- ⑥ 보고서 PDF 3개는 다운로드 매니페스트와 대조했다. ⑧은 제공된 ZIP 2개 속 PDF 18개의 해시를 색인 및 압축 해제본과 대조했다.
- ①~⑦ 포털 페이지의 이용조건을 자료별로 확인했다. ①·⑥은 출처표시·변경금지 조건이 표시되어 있어, 원문·청크·가공물의 외부 공유와 LLM 전송은 조건이 확인되기 전까지 공개 범위에 넣지 않는다. ②~⑤와 ⑦의 포털 표시는 이용허락범위 제한 없음이다. ⑥의 링크가 가리키는 개별 PDF와 ⑧ GUIDE의 이용 조건은 포털 목록 조건만으로 단정하지 않는다.

### 다음 단계에서 사용할 것

- M1은 ②~⑤의 원본 기준일 `2025-12-31`과 전체 응답을 보존하고, API 응답과 내려받은 파일을 다시 대조한다.
- M2는 ①의 두 업종 시트를 구분하고, ②~⑤는 대업종·규모·기준일이 맞는지 확인한 뒤 결합한다.
- RAG에서는 ⑥의 5행 목록 대신 확인된 실제 문서만 쓰고, 출처 URL·게시일·문서 ID를 청크마다 보존한다.

### 여기서 알게 된 점과 주의할 점

- 파일 다운로드 페이지의 자료번호와 실제 OpenAPI의 경로·UUID는 다르다. 재수집할 때는 M0의 API 경로와 자료별 키 변수명을 같이 확인한다.
- HTTP 200과 2행 응답은 API가 호출된다는 뜻일 뿐, 전체 페이지 수집이나 최신성·내용의 정확성을 보증하지 않는다. 그 검증은 M1에서 한다.
- ④ 사업장 수를 ⑤ 사망만인율의 분모로 쓰지 않는다. ⑤는 제공기관의 공식 지표로 보고 정의·기준연도를 그대로 표시한다.
- SIF 포털 설명에는 원자료 일부가 실제 재해 상황과 다를 수 있고 변경을 금지한다는 주의가 있다. SIF 문장을 임의로 고치거나, 감소대책 예시를 모든 현장의 정답처럼 쓰지 않는다.
- KOSHA GUIDE는 참고 기술지침으로 다룬다. 법령 자체와 같은 효력·최신성으로 표현하지 말고, 지침 번호와 개정연도를 함께 보여준다.

### M0 완료 여부

①~⑧의 자료 역할·파일 형식·출처와 이용조건을 정리했다. 2026-10-03 재검증에서 ②~⑥ 필수 API가 모두 HTTP 200으로 응답해 M0의 API 확인도 완료했다. ① 최신 게시본은 별도로 다운로드해 기존 로컬 파일과 SHA-256이 일치하는 것을 확인했다. M1에서는 API 전체 응답과 기존 원본 파일의 행·열을 대조한다.

### 확인한 공식 페이지

① [SIF 아카이브](https://www.data.go.kr/data/15140383/fileData.do) · ② [사고재해자수](https://www.data.go.kr/data/15084672/fileData.do) · ③ [사고사망자수](https://www.data.go.kr/data/15084674/fileData.do) · ④ [사업장수](https://www.data.go.kr/data/15064487/fileData.do) · ⑤ [사망만인율](https://www.data.go.kr/data/15064491/fileData.do) · ⑥ [사고정보·예방조치 자료](https://www.data.go.kr/data/15162988/fileData.do) · ⑦ [산업현장 법률용어 데이터](https://www.data.go.kr/data/15161288/fileData.do) · ⑧ [KOSHA GUIDE 공식 자료실](https://portal.kosha.or.kr/archive/resources/tech-support/search/all)
